# Notebook 07 — Construcción del dataset target para clasificación trimestral

A partir del consolidado LLM generado en el notebook 05 y validado en el notebook 06, este notebook construye el **dataset target final para clasificación** a nivel de trimestre.

En lugar de predecir un número exacto de picadas o capturas, el objetivo pasa a ser asignar a cada trimestre una **clase cualitativa de actividad pesquera** (`low`, `medium`, `high`). Esta reformulación está más alineada con el tamaño muestral disponible y con la orientación metodológica acordada para el TFG.

## Objetivo

- Cargar el dataset consolidado de extracciones LLM.
- Recuperar metadatos temporales del conjunto priorizado.
- Construir una base de trabajo a nivel de vídeo.
- Agregar la información por trimestre (`year_quarter`).
- Definir una etiqueta objetivo trimestral de clasificación.
- Validar la distribución de clases.
- Guardar el dataset target final para los notebooks posteriores de variables externas y modelado.

## Preparación del entorno

Montaje de Google Drive, carga de librerías base y definición de rutas.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path("/content/drive/MyDrive/PIDS4jjj2")
OUT_DIR = BASE_DIR / "outputs" / "llm_activity"

LLM_PATH = OUT_DIR / "llm_priority_dataset.parquet"
PRIORITY_PATH = OUT_DIR / "candidates_priority.parquet"

## Carga de los datasets base

Se carga el consolidado LLM del notebook 05 y el subconjunto priorizado generado en el notebook 04. El cruce por `video_id` permite recuperar la información temporal necesaria para construir el target trimestral.

In [ ]:
df_llm = pd.read_parquet(LLM_PATH)
df_priority = pd.read_parquet(PRIORITY_PATH)

print("Shape llm:", df_llm.shape)
print("Shape priority:", df_priority.shape)

Shape llm: (151, 8)
Shape priority: (151, 38)


In [ ]:
meta_cols = [
    "video_id",
    "title",
    "channel",
    "year",
    "year_quarter",
    "priority_score"
]

meta_cols_existing = [c for c in meta_cols if c in df_priority.columns]

df = df_llm.merge(
    df_priority[meta_cols_existing].drop_duplicates(subset=["video_id"]),
    on="video_id",
    how="left"
)

print("Shape tras merge:", df.shape)
df.head()

Shape tras merge: (151, 13)


,video_id,activity_mentions_llm,captures_count_llm,species_detected_llm,activity_level_llm,certainty_llm,evidence_llm,notes_llm,title,channel,year,year_quarter,priority_score
0,K2oTCf-5C4c,1,NaN,"[carpa, barbo, lucio, bass]",low,medium,salidas de pesca en varias modalidades y sobre...,"Se menciona actividad de pesca y modalidades, ...",rubenylolo......carpfishing 🤍🩵,RubenYlolo,2026.0,2026-Q2,21.0
1,KYkoHio90Pw,1,NaN,"[carpa, lucioperca]",high,high,Sorpresas con peces grandes en lagos urbanos d...,Hay actividad pesquera clara y capturas mencio...,Carpfishing 🥵 🔥Summer Campaign 🔥🥵 HOT FISHING🎣,Gerar Carp_angler,2026.0,2026-Q1,18.0
2,TvT1jlG9ayw,1,NaN,"[lucio, carpa]",low,medium,"se pueden recorrer en bici o a pie, hacer pira...",Vídeo descriptivo del entorno; menciona pesca ...,El embalse de Valmayor y su entorno,Infosierrademadrid.es,2016.0,2016-Q3,18.0
3,kt78DbVibg0,1,NaN,[carpa],high,high,LLUVIA y MUCHAS CAPTURAS; las carpas dieron la...,"Capturas claras, pero sin número exacto.",¡LLUVIA y MUCHAS CAPTURAS! 💥 Carp Fishing bajo...,RubenYlolo,2025.0,2025-Q4,18.0
4,dZfMo3t8TWo,1,NaN,[carpa],high,high,sesión increíble en solitario en valma con dob...,"Se mencionan picadas y carpas, pero no número ...",🔥Sesion increíble!! 🔥en solitario en valma con...,RubenYlolo,2025.0,2025-Q4,18.0


In [ ]:
print("Columnas disponibles:")
print(df.columns.tolist())

Columnas disponibles:
['video_id', 'activity_mentions_llm', 'captures_count_llm', 'species_detected_llm', 'activity_level_llm', 'certainty_llm', 'evidence_llm', 'notes_llm', 'title', 'channel', 'year', 'year_quarter', 'priority_score']


## Validación inicial

Se comprueba que el cruce ha recuperado correctamente las variables temporales y que el dataset está listo para agregarse por trimestre.

In [ ]:
check_cols = [c for c in ["video_id", "year", "year_quarter", "activity_mentions_llm", "activity_level_llm", "certainty_llm"] if c in df.columns]

print("Nulos por columna clave:")
display(df[check_cols].isna().sum())

Nulos por columna clave:


,0
video_id,0
year,2
year_quarter,2
activity_mentions_llm,0
activity_level_llm,0
certainty_llm,0


In [ ]:
print("Distribución por año:")
display(df["year"].value_counts().sort_index())

print("\nDistribución por trimestre:")
display(df["year_quarter"].value_counts().sort_index())

Distribución por año:


,count
year,
2009.0,3
2010.0,3
2011.0,6
2012.0,4
2013.0,7
2014.0,10
2015.0,2
2016.0,3
2017.0,7



Distribución por trimestre:


,count
year_quarter,
2009-Q3,2
2009-Q4,1
2010-Q1,1
2010-Q2,2
2011-Q1,1
2011-Q3,3
2011-Q4,2
2012-Q2,1
2012-Q3,2


## Normalización de señales LLM

Se preparan algunas variables auxiliares para facilitar la agregación trimestral. La idea es resumir no solo la clase cualitativa dominante, sino también la intensidad relativa de la señal en cada trimestre.

In [ ]:
df["activity_mentions_llm"] = df["activity_mentions_llm"].fillna(0).astype(int)

df["is_low"] = (df["activity_level_llm"] == "low").astype(int)
df["is_medium"] = (df["activity_level_llm"] == "medium").astype(int)
df["is_high"] = (df["activity_level_llm"] == "high").astype(int)

df["is_certainty_high"] = (df["certainty_llm"] == "high").astype(int)
df["is_certainty_medium"] = (df["certainty_llm"] == "medium").astype(int)
df["is_certainty_low"] = (df["certainty_llm"] == "low").astype(int)

df.head()

,video_id,activity_mentions_llm,captures_count_llm,species_detected_llm,activity_level_llm,certainty_llm,evidence_llm,notes_llm,title,channel,year,year_quarter,priority_score,is_low,is_medium,is_high,is_certainty_high,is_certainty_medium,is_certainty_low
0,K2oTCf-5C4c,1,NaN,"[carpa, barbo, lucio, bass]",low,medium,salidas de pesca en varias modalidades y sobre...,"Se menciona actividad de pesca y modalidades, ...",rubenylolo......carpfishing 🤍🩵,RubenYlolo,2026.0,2026-Q2,21.0,1,0,0,0,1,0
1,KYkoHio90Pw,1,NaN,"[carpa, lucioperca]",high,high,Sorpresas con peces grandes en lagos urbanos d...,Hay actividad pesquera clara y capturas mencio...,Carpfishing 🥵 🔥Summer Campaign 🔥🥵 HOT FISHING🎣,Gerar Carp_angler,2026.0,2026-Q1,18.0,0,0,1,1,0,0
2,TvT1jlG9ayw,1,NaN,"[lucio, carpa]",low,medium,"se pueden recorrer en bici o a pie, hacer pira...",Vídeo descriptivo del entorno; menciona pesca ...,El embalse de Valmayor y su entorno,Infosierrademadrid.es,2016.0,2016-Q3,18.0,1,0,0,0,1,0
3,kt78DbVibg0,1,NaN,[carpa],high,high,LLUVIA y MUCHAS CAPTURAS; las carpas dieron la...,"Capturas claras, pero sin número exacto.",¡LLUVIA y MUCHAS CAPTURAS! 💥 Carp Fishing bajo...,RubenYlolo,2025.0,2025-Q4,18.0,0,0,1,1,0,0
4,dZfMo3t8TWo,1,NaN,[carpa],high,high,sesión increíble en solitario en valma con dob...,"Se mencionan picadas y carpas, pero no número ...",🔥Sesion increíble!! 🔥en solitario en valma con...,RubenYlolo,2025.0,2025-Q4,18.0,0,0,1,1,0,0


## Agregación a nivel de trimestre

Se resume la información de todos los vídeos pertenecientes a un mismo trimestre. Esta será la unidad de análisis para el problema de clasificación.

In [ ]:
quarter_df = (
    df.groupby(["year", "year_quarter"], as_index=False)
    .agg(
        n_videos=("video_id", "count"),
        mean_activity_mentions=("activity_mentions_llm", "mean"),
        share_low=("is_low", "mean"),
        share_medium=("is_medium", "mean"),
        share_high=("is_high", "mean"),
        share_certainty_high=("is_certainty_high", "mean"),
        share_certainty_medium=("is_certainty_medium", "mean"),
        share_certainty_low=("is_certainty_low", "mean"),
        captures_nonnull_mean=("captures_count_llm", "mean")
    )
    .sort_values(["year", "year_quarter"])
    .reset_index(drop=True)
)

quarter_df.head(15)

,year,year_quarter,n_videos,mean_activity_mentions,share_low,share_medium,share_high,share_certainty_high,share_certainty_medium,share_certainty_low,captures_nonnull_mean
0,2009.0,2009-Q3,2,1.000000,1.000000,0.000000,0.000000,0.000000,0.500000,0.500000,NaN
1,2009.0,2009-Q4,1,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.0
2,2010.0,2010-Q1,1,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,NaN
3,2010.0,2010-Q2,2,1.000000,0.000000,1.000000,0.000000,0.500000,0.500000,0.000000,1.0
4,2011.0,2011-Q1,1,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.0
5,2011.0,2011-Q3,3,1.000000,0.000000,1.000000,0.000000,0.666667,0.333333,0.000000,1.0
6,2011.0,2011-Q4,2,1.000000,0.500000,0.500000,0.000000,0.500000,0.000000,0.500000,1.0
7,2012.0,2012-Q2,1,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,NaN
8,2012.0,2012-Q3,2,1.000000,1.000000,0.000000,0.000000,0.000000,0.500000,0.500000,NaN
9,2012.0,2012-Q4,1,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,1.0


In [ ]:
print("Shape trimestral:", quarter_df.shape)
quarter_df[["year_quarter", "n_videos", "share_low", "share_medium", "share_high"]].head(20)

Shape trimestral: (54, 11)


,year_quarter,n_videos,share_low,share_medium,share_high
0,2009-Q3,2,1.000000,0.000000,0.000000
1,2009-Q4,1,0.000000,1.000000,0.000000
2,2010-Q1,1,1.000000,0.000000,0.000000
3,2010-Q2,2,0.000000,1.000000,0.000000
4,2011-Q1,1,0.000000,1.000000,0.000000
5,2011-Q3,3,0.000000,1.000000,0.000000
6,2011-Q4,2,0.500000,0.500000,0.000000
7,2012-Q2,1,1.000000,0.000000,0.000000
8,2012-Q3,2,1.000000,0.000000,0.000000
9,2012-Q4,1,0.000000,1.000000,0.000000


## Definición de la clase objetivo trimestral

La clase objetivo se define a partir del nivel de actividad dominante en cada trimestre. Para evitar decisiones inestables, la asignación se basa en la mayor proporción entre `low`, `medium` y `high`.

In [ ]:
quarter_df["target_class"] = quarter_df[["share_low", "share_medium", "share_high"]].idxmax(axis=1)

quarter_df["target_class"] = quarter_df["target_class"].replace({
    "share_low": "low",
    "share_medium": "medium",
    "share_high": "high"
})

quarter_df[["year_quarter", "n_videos", "share_low", "share_medium", "share_high", "target_class"]].head(20)

,year_quarter,n_videos,share_low,share_medium,share_high,target_class
0,2009-Q3,2,1.000000,0.000000,0.000000,low
1,2009-Q4,1,0.000000,1.000000,0.000000,medium
2,2010-Q1,1,1.000000,0.000000,0.000000,low
3,2010-Q2,2,0.000000,1.000000,0.000000,medium
4,2011-Q1,1,0.000000,1.000000,0.000000,medium
5,2011-Q3,3,0.000000,1.000000,0.000000,medium
6,2011-Q4,2,0.500000,0.500000,0.000000,low
7,2012-Q2,1,1.000000,0.000000,0.000000,low
8,2012-Q3,2,1.000000,0.000000,0.000000,low
9,2012-Q4,1,0.000000,1.000000,0.000000,medium


## Variable auxiliar ordinal

Además de la etiqueta categórica, se construye una codificación ordinal que podrá ser útil en análisis descriptivos o pruebas complementarias.

In [ ]:
class_to_int = {"low": 0, "medium": 1, "high": 2}
quarter_df["target_class_id"] = quarter_df["target_class"].map(class_to_int)

quarter_df[["year_quarter", "target_class", "target_class_id"]].head(15)

,year_quarter,target_class,target_class_id
0,2009-Q3,low,0
1,2009-Q4,medium,1
2,2010-Q1,low,0
3,2010-Q2,medium,1
4,2011-Q1,medium,1
5,2011-Q3,medium,1
6,2011-Q4,low,0
7,2012-Q2,low,0
8,2012-Q3,low,0
9,2012-Q4,medium,1


## Distribución de clases

Se revisa si la variable objetivo queda razonablemente repartida entre clases. Esta comprobación es importante porque condicionará la viabilidad del modelado posterior.

In [ ]:
quarter_df["target_class"].value_counts()

,count
target_class,
low,28
medium,22
high,4


In [ ]:
quarter_df[["year_quarter", "n_videos", "target_class", "target_class_id", "share_low", "share_medium", "share_high", "share_certainty_high"]]

,year_quarter,n_videos,target_class,target_class_id,share_low,share_medium,share_high,share_certainty_high
0,2009-Q3,2,low,0,1.000000,0.000000,0.000000,0.000000
1,2009-Q4,1,medium,1,0.000000,1.000000,0.000000,1.000000
2,2010-Q1,1,low,0,1.000000,0.000000,0.000000,0.000000
3,2010-Q2,2,medium,1,0.000000,1.000000,0.000000,0.500000
4,2011-Q1,1,medium,1,0.000000,1.000000,0.000000,1.000000
5,2011-Q3,3,medium,1,0.000000,1.000000,0.000000,0.666667
6,2011-Q4,2,low,0,0.500000,0.500000,0.000000,0.500000
7,2012-Q2,1,low,0,1.000000,0.000000,0.000000,0.000000
8,2012-Q3,2,low,0,1.000000,0.000000,0.000000,0.000000
9,2012-Q4,1,medium,1,0.000000,1.000000,0.000000,0.000000


## Dataset final para notebooks posteriores

Se seleccionan las columnas que formarán el dataset target final para el cruce posterior con variables meteorológicas, hidrológicas y de presión.

In [ ]:
final_cols = [
    "year",
    "year_quarter",
    "n_videos",
    "mean_activity_mentions",
    "share_low",
    "share_medium",
    "share_high",
    "share_certainty_high",
    "captures_nonnull_mean",
    "target_class",
    "target_class_id"
]

target_dataset = quarter_df[final_cols].copy()

print("Shape final target dataset:", target_dataset.shape)
target_dataset.head(15)

Shape final target dataset: (54, 11)


,year,year_quarter,n_videos,mean_activity_mentions,share_low,share_medium,share_high,share_certainty_high,captures_nonnull_mean,target_class,target_class_id
0,2009.0,2009-Q3,2,1.000000,1.000000,0.000000,0.000000,0.000000,NaN,low,0
1,2009.0,2009-Q4,1,1.000000,0.000000,1.000000,0.000000,1.000000,1.0,medium,1
2,2010.0,2010-Q1,1,1.000000,1.000000,0.000000,0.000000,0.000000,NaN,low,0
3,2010.0,2010-Q2,2,1.000000,0.000000,1.000000,0.000000,0.500000,1.0,medium,1
4,2011.0,2011-Q1,1,1.000000,0.000000,1.000000,0.000000,1.000000,1.0,medium,1
5,2011.0,2011-Q3,3,1.000000,0.000000,1.000000,0.000000,0.666667,1.0,medium,1
6,2011.0,2011-Q4,2,1.000000,0.500000,0.500000,0.000000,0.500000,1.0,low,0
7,2012.0,2012-Q2,1,1.000000,1.000000,0.000000,0.000000,0.000000,NaN,low,0
8,2012.0,2012-Q3,2,1.000000,1.000000,0.000000,0.000000,0.000000,NaN,low,0
9,2012.0,2012-Q4,1,1.000000,0.000000,1.000000,0.000000,0.000000,1.0,medium,1


In [ ]:
TARGET_PARQUET = OUT_DIR / "valmayor_target_quarter_classification.parquet"
TARGET_CSV = OUT_DIR / "valmayor_target_quarter_classification.csv"

target_dataset.to_parquet(TARGET_PARQUET, index=False)
target_dataset.to_csv(TARGET_CSV, index=False, encoding="utf-8")

print("Guardado parquet:", TARGET_PARQUET)
print("Guardado csv:", TARGET_CSV)



Guardado parquet: /content/drive/MyDrive/PIDS4jjj2/outputs/llm_activity/valmayor_target_quarter_classification.parquet
Guardado csv: /content/drive/MyDrive/PIDS4jjj2/outputs/llm_activity/valmayor_target_quarter_classification.csv


In [ ]:
df_check = pd.read_parquet(TARGET_PARQUET)

print("Shape recargado:", df_check.shape)
df_check.head()

Shape recargado: (54, 11)


,year,year_quarter,n_videos,mean_activity_mentions,share_low,share_medium,share_high,share_certainty_high,captures_nonnull_mean,target_class,target_class_id
0,2009.0,2009-Q3,2,1.0,1.0,0.0,0.0,0.0,NaN,low,0
1,2009.0,2009-Q4,1,1.0,0.0,1.0,0.0,1.0,1.0,medium,1
2,2010.0,2010-Q1,1,1.0,1.0,0.0,0.0,0.0,NaN,low,0
3,2010.0,2010-Q2,2,1.0,0.0,1.0,0.0,0.5,1.0,medium,1
4,2011.0,2011-Q1,1,1.0,0.0,1.0,0.0,1.0,1.0,medium,1


## Conclusión

Este notebook deja construida la nueva versión del dataset target del TFG, ya no como problema de regresión a nivel de vídeo, sino como problema de clasificación a nivel trimestral.

La reformulación permite trabajar con una unidad temporal más razonable que la anual y con una variable objetivo más robusta para el tamaño muestral disponible. En lugar de intentar predecir un número exacto de picadas o capturas, el pipeline pasa a asignar a cada trimestre una clase de actividad (`low`, `medium`, `high`) basada en la señal agregada inferida por el LLM.

El resultado de este notebook será la base para integrar las variables externas en los notebooks siguientes y, posteriormente, entrenar modelos de clasificación más defendibles metodológicamente.